# Repeated `idcontrato` in the raw workbooks

`bulk-ingest` keeps one row per `idcontrato` (last wins). Until the 2026-11-15 republication a few
thousand ids appeared twice, because a multi-criteria contract was split over rows by
`TipoCriterioAdjudicacao`. The portal dropped that column and now exports each id once. This
notebook is the guard: it counts repeated ids in `data/raw/contracts_raw_<YEAR>.parquet` and fails
if any come back, so `keep="last"` stays a no-op safeguard and not a silent data loss.

In [1]:
from pathlib import Path

import polars as pl

from base_gov import raw_ingest

DATA = Path("..") / "data"
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_fmt_str_lengths(80)

years = raw_ingest.raw_years(DATA / "raw")
raw = pl.concat(
    [pl.read_parquet(raw_ingest.raw_path(DATA / "raw", y)).with_columns(src_year=pl.lit(y)) for y in years],
    how="diagonal_relaxed",
)
dups = raw.filter(pl.col("idcontrato").is_duplicated()).sort("idcontrato")
print(f"years={years}  raw rows={raw.height:,}  repeated rows={dups.height:,}  ids={dups['idcontrato'].n_unique():,}")

years=[2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]  raw rows=2,295,074  repeated rows=0  ids=0


## 1. Repeated ids per year

In [2]:
per_year = raw.group_by("src_year").agg(
    pl.len().alias("rows"),
    pl.col("idcontrato").is_duplicated().sum().alias("rows_in_repeated_ids"),
    pl.col("idcontrato").filter(pl.col("idcontrato").is_duplicated()).n_unique().alias("repeated_ids"),
).sort("src_year")
display(per_year)
display(dups.group_by("idcontrato").len()["len"].value_counts().sort("len"))
# Do any ids repeat ACROSS years (i.e. would `keep=last` pick by file order)?
cross = dups.group_by("idcontrato").agg(pl.col("src_year").n_unique().alias("n_years")).filter(pl.col("n_years") > 1)
print("ids spanning more than one year file:", cross.height)

src_year,rows,rows_in_repeated_ids,repeated_ids
i32,u32,u32,u32
2012,64783,0,0
2013,83217,0,0
2014,102516,0,0
2015,107636,0,0
2016,114897,0,0
2017,128583,0,0
2018,127883,0,0
2019,145502,0,0
2020,176493,0,0


len,count
u32,u32


ids spanning more than one year file: 0


## 2. Guard

Every `idcontrato` must appear exactly once across all raw years.

In [3]:
assert dups.is_empty(), f"{dups['idcontrato'].n_unique():,} repeated idcontrato in the raw files"
print("OK: every idcontrato appears once")

OK: every idcontrato appears once
